# TempoVul: Dataset Construction

Constructs the 400-sample TempoVul dataset from three source corpora: the Juliet Test Suite, Devign, and Big-Vul.

This notebook reproduces the sampling logic described in Section 3.3.1 of the paper. Source dataset provenance is derived from which schema columns are populated per row (Juliet: `File_Path`; Big-Vul: `CWE ID`; Devign: `Code`), confirmed against the corrected split reported in the paper: Juliet 220 samples, Big-Vul 139 samples, Devign 41 samples.

**Inputs required:** `bigvul_clean_for_transformer.csv`, `devign_master.csv`, `juliet_c_function_level.csv` (not included in this repository; available from their original sources, see paper Section 3.3.1 for citations).

## 1. Install dependencies and load source datasets

In [ ]:
!pip install pandas numpy -q

import pandas as pd
import numpy as np
import json

In [ ]:
bigvul_df = pd.read_csv('bigvul_clean_for_transformer.csv')
devign_df = pd.read_csv('devign_master.csv')
juliet_df = pd.read_csv('juliet_c_function_level.csv')

print("BIGVUL:", bigvul_df.shape)
print("DEVIGN:", devign_df.shape)
print("JULIET:", juliet_df.shape)

## 2. Check CWE distributions across source corpora

In [ ]:
print("BIGVUL CWE Distribution:")
print(bigvul_df['CWE ID'].value_counts().head(20))
print(f"Vulnerable: {bigvul_df['vul'].sum()}, Safe: {(bigvul_df['vul'] == 0).sum()}")

print()
print("JULIET CWE Distribution:")
juliet_df['CWE_numeric'] = juliet_df['CWE'].str.extract(r'(\d+)').astype(int)
print(juliet_df['CWE_numeric'].value_counts().head(20))
print(f"Vulnerable: {(juliet_df['Label'] == 1).sum()}, Safe: {(juliet_df['Label'] == 0).sum()}")

print()
print("DEVIGN Label Distribution:")
print(f"Vulnerable: {(devign_df['Label'] == 1).sum()}, Safe: {(devign_df['Label'] == 0).sum()}")
print("Note: Devign has no CWE labels, used for the General category.")

## 3. Strategic sampling by CWE category

Samples are drawn per category to match the target composition reported in Table 3 of the paper. Column names are standardized across the three source schemas first.

In [ ]:
bigvul_df['code'] = bigvul_df['func_before']
bigvul_df['label'] = bigvul_df['vul']
bigvul_df['cwe'] = bigvul_df['CWE ID'].str.extract(r'(\d+)')[0]
bigvul_df['cwe'] = pd.to_numeric(bigvul_df['cwe'], errors='coerce')

devign_df['code'] = devign_df['Code']
devign_df['label'] = devign_df['Label']
devign_df['cwe'] = None

juliet_df['code'] = juliet_df['Function_Code']
juliet_df['label'] = juliet_df['Label']
juliet_df['cwe'] = juliet_df['CWE_numeric']

def sample_cwe_category(df, cwe_list, n_vuln, n_safe, category_name):
    """Sample balanced vulnerable/safe rows from a given CWE list."""
    df_clean = df[df['cwe'].notna()].copy()
    df_clean['cwe'] = df_clean['cwe'].astype(int)
    cwe_mask = df_clean['cwe'].isin(cwe_list)

    vuln_samples = df_clean[cwe_mask & (df_clean['label'] == 1)]
    safe_samples = df_clean[cwe_mask & (df_clean['label'] == 0)]

    print(f"{category_name}: available {len(vuln_samples)} vuln, {len(safe_samples)} safe")

    if len(vuln_samples) < n_vuln or len(safe_samples) < n_safe:
        print(f"  Not enough samples, reducing target")
        n_vuln = min(n_vuln, len(vuln_samples))
        n_safe = min(n_safe, len(safe_samples))

    if n_vuln == 0 or n_safe == 0:
        print(f"  Skipping, no samples available")
        return pd.DataFrame()

    vuln = vuln_samples.sample(n=n_vuln, random_state=42)
    safe = safe_samples.sample(n=n_safe, random_state=42)
    result = pd.concat([vuln, safe])
    result['category'] = category_name

    print(f"  Sampled {len(vuln)} vuln, {len(safe)} safe")
    return result

In [ ]:
samples = []

# Memory Safety
memory_juliet = sample_cwe_category(juliet_df, [121, 122, 124, 127], 40, 40, "Memory Safety (Juliet)")
memory_bigvul = sample_cwe_category(bigvul_df, [119, 125, 787], 20, 20, "Memory Safety (BigVul)")
if not memory_juliet.empty: samples.append(memory_juliet)
if not memory_bigvul.empty: samples.append(memory_bigvul)

# Injection
injection = sample_cwe_category(juliet_df, [78, 89, 79], 40, 40, "Injection")
if not injection.empty: samples.append(injection)

# Numeric Errors
numeric = sample_cwe_category(juliet_df, [190, 191, 369], 30, 30, "Numeric Errors")
if not numeric.empty: samples.append(numeric)

# Resource Misuse is drawn from Big-Vul; see paper Section 3.3.1 for the
# full provenance attribution across all seven categories
resource = sample_cwe_category(bigvul_df, [416, 415, 404], 40, 40, "Resource Misuse")
if not resource.empty: samples.append(resource)

# Access Control: attempted from both Juliet and Big-Vul. In practice,
# Juliet did not yield qualifying samples for these CWEs, so the final
# Access Control category is Big-Vul only (19 samples total, not the
# intended 30), consistent with Table 3 in the paper.
access_juliet = sample_cwe_category(juliet_df, [862, 863, 306], 15, 15, "Access Control (Juliet)")
access_bigvul = sample_cwe_category(bigvul_df, [862, 863, 306], 15, 15, "Access Control (BigVul)")
if not access_juliet.empty: samples.append(access_juliet)
if not access_bigvul.empty: samples.append(access_bigvul)

tempovul_dataset = pd.concat(samples, ignore_index=True)

print()
print(f"Total so far: {len(tempovul_dataset)}")
print(f"Vulnerable: {(tempovul_dataset['label'] == 1).sum()}, Safe: {(tempovul_dataset['label'] == 0).sum()}")
print(tempovul_dataset['category'].value_counts())

## 4. Fill the remaining gap with Devign general samples

The categories above intentionally do not sum to 400. The remainder is filled from Devign to produce the General category.

In [ ]:
needed = 400 - len(tempovul_dataset)
needed_vuln = needed // 2
needed_safe = needed - needed_vuln

print(f"Sampling {needed_vuln} vuln + {needed_safe} safe from Devign")

devign_vuln = devign_df[devign_df['label'] == 1].sample(n=needed_vuln, random_state=42)
devign_safe = devign_df[devign_df['label'] == 0].sample(n=needed_safe, random_state=42)

devign_samples = pd.concat([devign_vuln, devign_safe])
devign_samples['category'] = 'General (Devign)'

for col in tempovul_dataset.columns:
    if col not in devign_samples.columns:
        devign_samples[col] = None

tempovul_final = pd.concat([tempovul_dataset, devign_samples], ignore_index=True)

print()
print(f"Total samples: {len(tempovul_final)}")
print(f"Vulnerable: {(tempovul_final['label'] == 1).sum()}")
print(f"Safe: {(tempovul_final['label'] == 0).sum()}")
print()
print(tempovul_final['category'].value_counts())

tempovul_final.to_csv('tempovul_base_dataset.csv', index=False)
print()
print("Saved: tempovul_base_dataset.csv")

Next: see `02_artifact_and_evaluation_pipeline.ipynb` to generate the four stage-specific artifacts and run LLM evaluation.